<a href="https://colab.research.google.com/github/Haidriyam/statutory-semantic-intertia/blob/main/legal_inertia_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# Diachronic Vector Space Alignment and Displacement Demo (Reference Implementation)
# Repository: statutory-semantic-intertia
# ==============================================================================

# 1. Environment & Dependency Setup
import sys
import subprocess

try:
    import gensim
    import scipy
    import matplotlib
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gensim", "scipy", "matplotlib", "numpy"], check=True)

import numpy as np
from scipy.spatial.distance import cosine
from gensim.models import Word2Vec
import matplotlib.pyplot as plt

# Set random seeds for reproducible demonstration
np.random.seed(42)

# ==============================================================================
# 2. Synthetic Corpus Partitions (Controlled Illustrative Draws)
# ==============================================================================
# Epoch 1: Historical baseline framing (statutory anchors & period vernacular)
corpus_epoch1 = [
    ["the", "statute", "provides", "remedy", "hereinabove", "specified", "in", "the", "indenture"],
    ["the", "court", "decrees", "relief", "witnesseth", "lawful", "execution", "hereunto"],
    ["a", "considerable", "hazard", "threatened", "the", "shipping", "enterprise"],
    ["the", "nice", "distinction", "required", "exact", "calculation", "and", "judgment"],
    ["severe", "penalties", "attach", "to", "wilful", "default", "under", "seal"]
] * 35

# Epoch 2: Evolved framing
# 'hazard' and 'nice' shift distributional contexts; statutory terms remain fixed
corpus_epoch2 = [
    ["the", "statute", "provides", "remedy", "hereinabove", "specified", "in", "the", "indenture"],
    ["the", "court", "decrees", "relief", "witnesseth", "lawful", "execution", "hereunto"],
    ["an", "environmental", "hazard", "created", "toxic", "waste", "contamination"],
    ["they", "had", "a", "nice", "pleasant", "dinner", "conversation"],
    ["severe", "penalties", "attach", "to", "wilful", "default", "under", "seal"]
] * 35

# ==============================================================================
# 3. Model Training (Skip-Gram Architecture)
# ==============================================================================
print("[*] Training period models (Skip-gram, d=50)...")
model_t1 = Word2Vec(
    sentences=corpus_epoch1,
    vector_size=50,
    window=4,
    min_count=2,
    sg=1,
    seed=42,
    workers=1,
    epochs=10
)

model_t2 = Word2Vec(
    sentences=corpus_epoch2,
    vector_size=50,
    window=4,
    min_count=2,
    sg=1,
    seed=42,
    workers=1,
    epochs=10
)

# ==============================================================================
# 4. Orthogonal Procrustes Alignment via SVD
# ==============================================================================
shared_vocab = sorted(
    list(set(model_t1.wv.index_to_key).intersection(set(model_t2.wv.index_to_key)))
)

# Extract shared embeddings
w_base = np.array([model_t1.wv[w] for w in shared_vocab], dtype=float)
w_target = np.array([model_t2.wv[w] for w in shared_vocab], dtype=float)

# Row-normalize prior to SVD alignment
w_base_norm = w_base / np.linalg.norm(w_base, axis=1, keepdims=True)
w_target_norm = w_target / np.linalg.norm(w_target, axis=1, keepdims=True)

# Cross-covariance matrix M = W_target^T @ W_base
correlation_matrix = np.dot(w_target_norm.T, w_base_norm)
u, _, vt = np.linalg.svd(correlation_matrix, full_matrices=False)
q_rotation = np.dot(u, vt)

# Project target epoch space
aligned_target = {
    w: np.dot(model_t2.wv[w].astype(float), q_rotation)
    for w in model_t2.wv.index_to_key
}

# ==============================================================================
# 5. Metric Computation: Displacement & Neighborhood Overlap
# ==============================================================================
k_neighbors = 3
test_probe_words = ["hereinabove", "indenture", "witnesseth", "statute", "hazard", "nice"]

# Precompute normalized candidate matrices for matrix-vector lookups
candidate_words = shared_vocab
mat_c1 = np.array([model_t1.wv[w] for w in candidate_words], dtype=float)
mat_c2 = np.array([aligned_target[w] for w in candidate_words], dtype=float)

mat_c1 /= np.linalg.norm(mat_c1, axis=1, keepdims=True)
mat_c2 /= np.linalg.norm(mat_c2, axis=1, keepdims=True)

cand_to_idx = {w: i for i, w in enumerate(candidate_words)}

records = []
for word in test_probe_words:
    if word not in cand_to_idx:
        continue

    idx = cand_to_idx[word]
    v1 = model_t1.wv[word].astype(float)
    v2 = aligned_target[word].astype(float)

    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    cos_sim = float(np.dot(v1, v2) / (norm_v1 * norm_v2))
    cos_sim = float(np.clip(cos_sim, -1.0, 1.0))
    disp = 1.0 - cos_sim

    # Nearest neighbors in base epoch
    sims_t1 = np.dot(mat_c1, (v1 / norm_v1))
    sims_t1[idx] = -np.inf
    top_t1_idx = np.argpartition(sims_t1, -k_neighbors)[-k_neighbors:]
    n_t1 = {candidate_words[i] for i in top_t1_idx}

    # Nearest neighbors in aligned target epoch
    sims_t2 = np.dot(mat_c2, (v2 / norm_v2))
    sims_t2[idx] = -np.inf
    top_t2_idx = np.argpartition(sims_t2, -k_neighbors)[-k_neighbors:]
    n_t2 = {candidate_words[i] for i in top_t2_idx}

    # Jaccard stability
    union_n = n_t1.union(n_t2)
    jaccard = len(n_t1.intersection(n_t2)) / float(len(union_n)) if union_n else 0.0

    # Exploratory legacy composite
    lsi_exploratory = (1.0 - disp) * (0.5 + 0.5 * jaccard)

    records.append({
        "word": word,
        "displacement": disp,
        "jaccard": jaccard,
        "lsi_exploratory": lsi_exploratory
    })

# Sort ascending by displacement (most stable tokens first)
records.sort(key=lambda x: x["displacement"])

# ==============================================================================
# 6. Tabular Summary
# ==============================================================================
print("\n" + "=" * 65)
print(f"{'Probe Token':<16}{'Displacement (D)':<20}{'Jaccard (J)':<14}{'Legacy LSI':<12}")
print("=" * 65)
for r in records:
    print(f"{r['word']:<16}{r['displacement']:<20.4f}{r['jaccard']:<14.2f}{r['lsi_exploratory']:<12.4f}")
print("=" * 65)
print("[!] Note: Metrics derived on synthetic demonstration data.")

# ==============================================================================
# 7. Visualization Artifact
# ==============================================================================
words = [r["word"] for r in records]
displacements = [r["displacement"] for r in records]

plt.figure(figsize=(9, 4.5), dpi=150)
y_pos = np.arange(len(words))
colors = ["#1f4e5b" if d < 0.20 else "#b33939" for d in displacements]

plt.barh(y_pos, displacements, color=colors, edgecolor="black", linewidth=0.6)
plt.yticks(y_pos, words, fontsize=9)
plt.xlabel("Cosine Displacement $D(w) = 1 - \\cos(v_{t1}, Q^T v_{t2})$", fontsize=10)
plt.title("Diachronic Lexical Displacement (Reference Demonstration)", fontsize=11, fontweight="bold", pad=10)
plt.gca().invert_yaxis()
plt.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()